# Ch. 5: Array-Based Sequences

## 5.1: Python's Sequence Types
- sequence classes
    - list
    - tuple
    - str
- commonalities of classes
    - supports indexing to access individual element of sequence w/ syntax seq[k]
    - uses low-level concept of **array** to represent sequence
- significant differences in
    - abstractions the classes represent
    - the way the instances of the classes are represented internally in python
- sequence classes are widely used + building blocks for complex data structures
    - must understand their public behavior + inner workings

**Public Behaviors**
- important subleties regarding behaviors associated with these classes

**Implementation Details**
- **encapsulation** --> user of class doesn't need to know about internal details of implementation
- but it's important to understand implementation details bc efficiency of program depends on efficiency of components on which it relies

**Asymptotic and Experimental Analyses**
- rely on formal asymptotic analysis
- perform experimental analyses of primary operations to provide empriical evidence consistent with more theoretical asymptotic analyses

## 5.2: Low-Level Arrays

Low-level computer architecture
- primary memory of computer is composed of bits of information
    - bits are grouped into larger units tha depend on precise sytem architecture
    - typically, a unit is a **byte** (8 bits)
- computer system has huge number of bytes of memory
- **memory addresses** = allow computer to keep track of what information is stored at what byte
    - each byte is associated with unique number (its address)
        - address = binary representation of the number
    - memory addresses are typically coordinated with physical layout of memory system, so we can often portray numbers in sequential fashion

## 5.3: Dynamic Arrays and Amortization

In [ ]:
# Implementation of DynamicArray class using a raw array from ctypes module as storage
import ctypes # provides support for low-level arrays
class DynamicArray:
    '''A dynamic array class akin to a simplified Python list.'''
    def __init__(self):
        '''Create an empty array.'''
        self._n = 0 # actual elements in array
        self._capacity = 1 # default array capacity
        self._A = self._make_array(self._capacity) # low-level array

    def __len__(self):
        '''Return number of elements stored in the array.'''
        return self._n

    def __getitem__(self, k):
        '''Return element at index k'''
        if not 0 <= k < self._n:
            raise IndexError('invalid index')
        return self._A[k] # retrieve from array

    def append(self, obj):
        '''Add object to end of array'''
        if self._n == self._capacity: # already at capacity, can't add another element without resizing
            # resize
            self._resize(2 * self._capacity) # double capacity
        self._A[self._n] = obj
        self._n += 1

    def _resize(self, c): # nonpublic utility
        '''Resize internal array to capacity c.'''
        B = self._make_array(c)
        for k in range(self._n): # assign each element in B to corresponding element in A
            B[k] = self._A[k]
        self._A = B # set the array to this new array
        self._capacity = c


    def _make_array(self, c): # nonpublic utility
        '''Return new array with capacity c'''
        return (c * ctypes.py_object)() # see ctypes documentation


## 5.5: Using Array-Based Sequences

### 5.5.1: Storing high Scores for.a Game
- many applications where want to store a sequence of objects
- info to include an object representing a high score entry
    - integer representing score itself: _score
    - name of person earning score: _name
    - date?
    - game statistics?


In [ ]:
class GameEntry:
    def __init__(self, score, name):
        self._score = score
        self._name = name

    def get_name(self):
        return self._name

    def get_score(self):
        return self._score

    # return string representation of this score entry
    def __str__(self): # called automatically when do print(obj)
        return '({0}, {1})'.format(self._name, self._score) # e.g. '(Bob, 98)'
            # {0} replaced by 0th argument, {1} replaced by 1st argument
            # now more commonly use f'({self._name}, {self._score})'

#### A Class for High Scores
- to maintain sequence of high scores, make class Scoreboard
    - limited number of high scores
    - once limit is reached, new score only qualifies for scoreboard if strictly higher than lowest 'highscore' on board
- limit of scores depends on game -> speciy as a parameter in Scoreboard constructor
- internally, use python list named _board to manage GameEntry instances that represent the high scores
    - initialize list to be large enough to hold maximum number of scores, but initially set all entries to None
    - allocate list with max capacity initially -> never needs to be resized
    - add entries from highest to lowest score
    - store references to GameEntry objects
- self._board = [None] * capacity
    - creates list of desired length with all entries equal to None
- self._n = instance variable to represent number of actual entries currently in table
- _ _ getitem _ _ method to retrieve an entry at a given index with syntax board[i] (or return None if no such entry exists)
    - bc the object Scoreboard contains a list isn't a list itself, so we need to define _ _ getitem _ _ to support indexing
- _ _ str _ _ method to return string representation of entire scoreboard, with one entry per line
    - so we can call len(scoreboard_instance)

In [ ]:
class Scoreboard:
    '''Fixed-length sequence of high scores in nondecreasing order.'''

    def __init__(self, capacity=10):
        '''Initialize scoreboard with given maximum capacity

        All entries are initially none
        '''
        self._board = [None] * capacity
        self._n = 0 # number scores initially is 0

    def __getitem__(self, k):
        '''Return entry at index k'''
        if k < self._n:
            return self._board[k]
        else:
            return None

    def __str__(self):
        '''Return string representation of the high score list.'''
        return '\n'.join(str(self._board[j]) for j in range(self._n)) # only the actual scores in the list

    def add(self, entry):
        '''Consider adding entry to high scores.'''
        score = entry.get_score() # entry is of type GameEntry

        # Does new entry quality as a high score
        # answer is yes if baord not full or score higher than last entry
        good = self._n < len(self._board) or score > self._board[-1].get_score() # put this condition first so it short circuits properly

        if good:
            if self._n < len(self._board):
                self._n += 1 # overall number of scores increases

        # shift lower scores rightward to make room for new entry
        j = self._n - 1 # start at last score
            # if n was incremented previously, self._n - 1 is empty
        while j > 0 and self._board[j-1].get_score() < score:
            self._board[j] = self._board[j-1] # shift entry from j-1 to j when entry j-1 is less than score we want to add
            j -= 1
        self._board[j] = entry #

#### Adding an Entry
- if board not yet full, any new entry retained
- once board is full, new entry is only retained if strictly better than one of the other scores
    - which rewquires it to be strictly better than last entry of scoreboard, which is the lowest of the high scores
- consider new score
    - determine if qualifies as high score
    - if yes, increase count of active scores _n unless board at full capacity
        - if at full capacity, adding a new high score causes some other entry to be dropped from scoreboard -> _n entries remains same
    - correctly place new entry within list
        - shift inferior scores one spot lower
            - least score drpped entirely when scoreboard full
        - similar to insert method of list class
        - no need to shift None referneces near end
    - consider index j = self._n - 1
        - index of which last GameEntry instance wil reside
        - either j is correct index for newest entry, or one or more scores before it are less than the new score
        - while loop checks compound condition + shifts references rightward + decrements j as long as there is another entry at index j-1 with score less than new score

### 5.5.2: Sorting a Sequence
- sorting: start with unordered sequence of elements, rearrange them into nondecreasing order

#### The Insertion-Sort Algorithm
- insertion-sort
    - start with 1st element
    - one element by itself is already sorted
    - consider 2nd element
        - if smaller than first, swap them
    - consider 3rd element
        - swap it leftward until in proper order with the first 2 elements
    - consider 4th element
        - swap it leftward until in proper order with first 3 elements
    - continue until whole array sorted

**Algoirthm** InsertionSort(A)
    **Input**: An array A of n comparable elements
    **Output**: The array A with elements rearranged in nondecreasing order
    **for** k from 1 to n-1 **do**
        Insert A[k] at its proper location within A[0], A[1], ..., A[k]

- task of inserting new entry into list of high scores is almost identical to task of inserting a newly considered element in insertion-sort
- python implementation
    - outer loop to consider each element in turn
    - inner loop that moves a newly considered element to its proper location relative to sorted subarray of elements to its left
- nested loops -> O(n^2) running time in worst case
    - most work done if array is initially in reverse order
    - if initial array is nearly sorted or perfectly sorted, runs in O(n) time bc few or no iterations of outer loop
- sorts in-place

In [ ]:
def insertion_sort(A):
    '''Sort list of comparable elements into nondecreasing order.'''
    for k in range(1, len(A)): # from 1 to n-1
        cur = A[k]
        j = k
        while j > 0 and A[j-1] > cur:
            A[j] = A[j-1]
            j -= 1
        A[j] = cur

### 5.5.3: Simple Cryptography
- **cryptography** = the science of secret messages + their applications
    - application of strings and lists
    - studies way of performing **encryption** (take a message called the **plaintext** and convert it into a scrambled message calledd the **ciphertext**)
    - studies corresponding ways of **decryption** (take a ciphertext and turn it back into the original plaintext)
- **Caesar cipher** = earliest encryption scheme
    - Julius Caesar used it to protetx important military messages
    - simple way to obscure a message written in a language that forms words with an alphabet
- Caesar cipher method
    - replace each letter in a message with the letter that is a certain number of letters after it in the alphabet
    - e.g. A -> D, B -> E, C -> F, ... to shift 3 characters
    - substititution pattern then **wraps around**
        - X -> A, Y -> B, Z -> C

#### Converting Between Strings and Character Lists
- strings are immutable -> can't directly edit an instance of string to encrupt it
- goal: generate new string
- technique for performing string transformations:
    - create equivalent list of characters `list('bird') -> ['b','i','r','d']`
    - edit list
    - reassemble a new string based on list `''.join(['b','i','r','d'])`


#### Using Characters as Array Indices
- number letters like array indices (A=0, B=1, C=2, ...)
- then can write Caesar cipher with a rotation r as a simple forumular
    - replace each letter i with the letter (i + r) mod 26
        - mod = **modulo** operator = returns remiander after performing integer division
        - allows us to perform arap around at end of alphabet
        - e.g. 27 mod 26 is 1
- decryption algorithm
    - replace each letter with the one r places before it (i -> (i - r) mod 26)
- represent replacement rule: use another string to describe the translation
    - precompute string that represents replacements that should be used
    - e.g. 3-char rotation: 'ABC...' -> 'DEF...'
    - use translation string as a guide to encrypt a message
    - challenge: quickly locate replacement for each character of original message
- chars are represented in Unicode by integer code points
    - code points are uppercase letters of latin alphabet are consecutive (we restrict are encryption to uppercase letters)
    - python supports functions that convert btwn integer code points <-> one-character strings
    - ord(c) takes a one-character string as parameter + returns integer code point for that character
    - chr(j) takes integer as parameter + returns its associated one-character string
- method for finding a replacement for a character
    - map A,...,Z to respective numbers 0,...,25
    - formula: j = ord(c) - ord('A')
    - the integer j can be used as an index into our precomputed translation string
- creating class for performing Caesar cipher with arbitrary rotational shift
    - builds forward + backward translation strings for given rotation
    - encryption + decryption algs are essentially the same
        - perform both by means of nonpublic utility method _transform


In [ ]:
class CaesarCipher:
    '''Class for doing encryption + decryption using a Caesar cipher.'''

    def __init__(self, shift):
        '''Construct Caesar cipher using given integer shift for rotation.'''
        encoder = [None] * 26 # temp array for encryption
        decoder = [None] * 26 # temp array for decryption
        for k in range(26):
            encoder[k] = chr((k + shift) % 26 + ord('A'))
            decoder[k] = chr((k - shift) % 26 + ord('A'))
        self._forward = ''.join(encoder) # store as string since fixed
        self._backward = ''.join(decoder)

    def encrypt(self, message):
        '''Return string representing encrypted message.'''
        return self._transform(message, self._forward)

    def decrypt(self, secret):
        '''Return decrypted message given encrypted secret.'''
        return self._transform(secret, self._backward)

    def _transform(self, original, code):
        '''Utility to perform transformation based on given code string.'''
        msg = list(original)
        for k in range(len(msg)):
            if msg[k].isupper():
                j = ord(msg[k]) - ord('A') # get number representing the character
                msg[k] = code[j]
        return ''.join(msg)

    # run the code
    if __name__ = '__main__':
        cipher = CaesarCipher(3)
        message = "THE EAGLE IS IN PLAY; MEET AT JOE'S."
        coded = cipher.encrypt(message)
        print('Secret: ', coded)
        answer = cipher.decrypt(coded)
        print('Message: ', answer)

## 5.6: Multidimensional Data Sets